In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import os
from dotenv import load_dotenv
import logging


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [3]:
load_dotenv("../.env")

True

In [4]:
DB_URL = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=os.getenv("SUPABASE_PASSWORD"),
    host="db.vqssaigndessrttplwib.supabase.co",
    port=5432,
    database="postgres"
)

In [5]:
engine = create_engine(DB_URL)
df = pd.read_sql("SELECT * FROM weather_raw", engine)
logger.info(f"Loaded {len(df)} rows from weather_raw")


2026-05-02 22:09:09,364 - INFO - Loaded 5 rows from weather_raw


In [7]:
df.head(3)

,time,interval,temperature,windspeed,winddirection,is_day,weathercode,fetched_at
0,2026-04-30T12:00,900,7.1,12.0,351,1,2,2026-04-30 12:12:18.842896
1,2026-04-30T12:15,900,6.8,11.6,352,0,2,2026-04-30 12:28:02.784784
2,2026-04-30T12:15,900,6.8,11.6,352,0,2,2026-04-30 12:29:05.720791


In [11]:
df.head()

,temperature,windspeed,winddirection,is_day,weathercode,fetched_at
0,7.1,12.0,351,1,2,2026-04-30 12:12:18.842896
1,6.8,11.6,352,0,2,2026-04-30 12:28:02.784784
2,6.8,11.6,352,0,2,2026-04-30 12:29:05.720791
3,8.0,2.2,104,0,1,2026-05-02 13:44:10.213465
4,7.8,2.9,112,0,2,2026-05-02 13:45:13.405585


In [ ]:
df["fetched_date"] = pd.to_datetime(df["fetched_at"]).dt.date

In [14]:
logger.info("Added fetched_time and fetched_hour")
df["fetched_time"] = pd.to_datetime(df["fetched_at"]).dt.strftime("%H:%M:%S")

2026-05-02 22:24:05,365 - INFO - Added fetched_time and fetched_hour


In [15]:
df.head()

,temperature,windspeed,winddirection,is_day,weathercode,fetched_at,fetched_date,fetched_hour,fetched_time
0,7.1,12.0,351,1,2,2026-04-30 12:12:18.842896,2026-04-30,12,12:12:18
1,6.8,11.6,352,0,2,2026-04-30 12:28:02.784784,2026-04-30,12,12:28:02
2,6.8,11.6,352,0,2,2026-04-30 12:29:05.720791,2026-04-30,12,12:29:05
3,8.0,2.2,104,0,1,2026-05-02 13:44:10.213465,2026-05-02,13,13:44:10
4,7.8,2.9,112,0,2,2026-05-02 13:45:13.405585,2026-05-02,13,13:45:13


In [17]:
logger.info("Removed columns: fetched_at, fetched_hour")
df.drop(columns=["fetched_at", "fetched_hour"], inplace=True)

2026-05-02 22:25:44,684 - INFO - Removed columns: fetched_at, fetched_hour


In [18]:
df.head(3)

,temperature,windspeed,winddirection,is_day,weathercode,fetched_date,fetched_time
0,7.1,12.0,351,1,2,2026-04-30,12:12:18
1,6.8,11.6,352,0,2,2026-04-30,12:28:02
2,6.8,11.6,352,0,2,2026-04-30,12:29:05


In [19]:
logger.info("Added new column called: weather_description")
weather_codes = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast"
}

df["weather_description"] = df["weathercode"].map(weather_codes)

2026-05-02 22:27:39,494 - INFO - Added new column called: weather_description


In [20]:
df.head(3)

,temperature,windspeed,winddirection,is_day,weathercode,fetched_date,fetched_time,weather_description
0,7.1,12.0,351,1,2,2026-04-30,12:12:18,Partly cloudy
1,6.8,11.6,352,0,2,2026-04-30,12:28:02,Partly cloudy
2,6.8,11.6,352,0,2,2026-04-30,12:29:05,Partly cloudy


In [23]:
logger.info("Ordering column names.")
df = df[["weather_description", "temperature", "is_day", "windspeed", "weathercode", "winddirection", "fetched_time", "fetched_date"]]

2026-05-02 22:31:27,603 - INFO - Ordering column names.


In [24]:
df.head(3)

,weather_description,temperature,is_day,windspeed,weathercode,winddirection,fetched_time,fetched_date
0,Partly cloudy,7.1,1,12.0,2,351,12:12:18,2026-04-30
1,Partly cloudy,6.8,0,11.6,2,352,12:28:02,2026-04-30
2,Partly cloudy,6.8,0,11.6,2,352,12:29:05,2026-04-30


In [25]:
logger.info("Writing to database.")
df.to_sql("weather_mart", engine, if_exists="replace", index=False)
logger.info(f"Wrote {len(df)} rows to weather_mart")

2026-05-02 22:35:57,640 - INFO - Writing to database.
2026-05-02 22:35:58,155 - INFO - Wrote 5 rows to weather_mart
